# 4주차: 나만의 신경망 만들기

## 오늘 배울 것 (4단계)
1. **Linear** : 점수를 매기는 심사위원
2. **ReLU** : 음수를 끄는 스위치
3. **설계도** : 심사위원과 스위치를 쌓아서 신경망 만들기
4. **학습** : 지난주와 똑같은 5단계로 학습시키기

> 오늘의 한 문장: 신경망은 **심사위원(Linear)** 과 **스위치(ReLU)** 를 차례로 쌓은 것이다.


## 준비: 도구와 데이터 불러오기

아래 셀 3개를 **위에서부터 순서대로** 실행하세요.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from torch.utils.data import TensorDataset, DataLoader, random_split

In [ ]:
digits = load_digits()
X = torch.tensor(digits.images, dtype=torch.float32) / 16.0
y = torch.tensor(digits.target)

In [ ]:
torch.manual_seed(0)
dataset = TensorDataset(X, y)
train_data, test_data = random_split(dataset, [1500, 297])
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

## 1단계. Linear: 점수를 매기는 심사위원

`nn.Linear`는 **특징마다 중요도를 곱해서 더하고, 기본 점수를 더하는** 심사위원입니다.

과일의 특징 3개를 보고 **사과 점수**와 **바나나 점수**를 매겨 봅시다.

| | 빨간 정도 | 노란 정도 | 길쭉한 정도 | 기본 점수 |
|---|---|---|---|---|
| 사과 심사위원 | +2 | −1 | −2 | 0 |
| 바나나 심사위원 | −1 | +2 | +2 | 0 |


In [ ]:
judge = nn.Linear(3, 2)
print(judge)

In [ ]:
judge.weight.data = torch.tensor([[ 2.0, -1.0, -2.0],
                                  [-1.0,  2.0,  2.0]])
judge.bias.data = torch.tensor([0.0, 0.0])

**실행 전에 손으로 계산해 보세요.** 빨갛고 둥근 과일 = [빨강 1, 노랑 0, 길쭉 0]

- 사과 점수 = 2×1 + (−1)×0 + (−2)×0 + 0 = ?
- 바나나 점수 = (−1)×1 + 2×0 + 2×0 + 0 = ?


In [ ]:
fruit = torch.tensor([1.0, 0.0, 0.0])
print(judge(fruit))

**직접 해 보기**: 노랗고 길쭉한 과일의 특징을 넣어 보세요. 실행 전에 점수를 예상해 보세요.


In [ ]:
fruit2 = torch.tensor([???, ???, ???])
print(judge(fruit2))

## 2단계. ReLU: 음수를 끄는 스위치

ReLU의 규칙은 딱 하나입니다. **음수는 0으로, 양수는 그대로.**

실행 전에 예상해 보세요. `[-3, -1, 0, 2, 5]` → ReLU → `[?, ?, ?, ?, ?]`


In [ ]:
relu = nn.ReLU()
numbers = torch.tensor([-3.0, -1.0, 0.0, 2.0, 5.0])
print(???)

## 3단계. 신경망 설계도 만들기

레고처럼 부품을 쌓아 **나만의 신경망**을 만듭니다.

```text
이미지(픽셀 64개) → [심사위원 64→32] → [스위치 ReLU] → [심사위원 32→10] → 숫자 점수 10개
```

설계도는 두 부분입니다.
- `__init__` : **부품 준비** (레고 블록 꺼내 놓기)
- `forward` : **데이터가 지나가는 순서** (블록 조립 순서)

빈칸 2곳을 위 그림의 순서대로 채우세요.


In [ ]:
class MyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(64, 32)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(32, 10)

    def forward(self, x):
        x = self.layer1(x)
        x = ???
        x = ???
        return x

In [ ]:
model = MyNet()
print(model)

모델에 이미지 32장을 넣어 봅시다. 마지막 결과의 모양(shape)을 예상해 보세요. `[?, ?]`


In [ ]:
images, labels = next(iter(train_loader))
scores = model(images.reshape(-1, 64))
print(scores.shape)

## 4단계. 학습시키기

학습 방법은 **지난주와 똑같습니다.** 모델만 바뀌었습니다.

| 단계 | 코드 | 하는 일 |
|---|---|---|
| 1 | `optimizer.zero_grad()` | 지난번 gradient 지우기 |
| 2 | `scores = model(...)` | 예측하기 |
| 3 | `loss = loss_fn(...)` | 얼마나 틀렸는지 오답 점수 계산 |
| 4 | `loss.backward()` | 어느 방향으로 고칠지 계산 |
| 5 | `optimizer.step()` | 실제로 중요도 고치기 |


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

In [ ]:
for epoch in range(10):
    total_loss = 0
    for images, labels in train_loader:
        optimizer.zero_grad()
        scores = model(images.reshape(-1, 64))
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print('epoch', epoch + 1, 'loss', round(total_loss / len(train_loader), 3))

## 결과 확인: 시험 보기

공부할 때 보지 않은 시험용 297장으로 정확도를 확인합니다.


In [ ]:
test_images, test_labels = test_data[:]
predictions = model(test_images.reshape(-1, 64)).argmax(dim=1)
accuracy = (predictions == test_labels).float().mean()
print('정확도:', round(accuracy.item() * 100, 1), '%')

`index`의 숫자를 0~296 사이에서 바꿔 가며 실행해 보세요. 모델이 **틀린 이미지**를 찾아보세요.


In [ ]:
index = 0
plt.imshow(test_images[index], cmap='gray')
plt.title(f'answer {test_labels[index].item()} / model {predictions[index].item()}')
plt.show()

## 오늘 정리

자신의 말로 답해 보세요.

1. Linear(심사위원)는 어떤 계산을 하나요?
2. ReLU(스위치)의 규칙은 무엇인가요?
3. 설계도에서 `__init__`과 `forward`는 각각 무슨 일을 하나요?
4. `loss.backward()`와 `optimizer.step()`은 무엇이 다른가요?

내 답:
- 1.
- 2.
- 3.
- 4.


## 도전 과제 (시간이 남으면)

### 도전 1. 중간 심사위원 수 바꾸기
셀 9에서 `32`를 **두 군데 모두** `4`로 바꾸고, 셀 9 → 10 → 12 → 13 → 14 순서로 다시 실행해 보세요. 정확도가 어떻게 되나요? `128`로도 해 보세요.

| 중간 개수 | 정확도 |
|---|---|
| 4 | |
| 32 | |
| 128 | |

### 도전 2. 스위치 빼 보기
셀 9의 forward에서 `x = self.relu(x)` 줄 앞에 `#`을 붙여 끄고 다시 학습시켜 보세요. 결과를 예상해 보세요.

### 도전 3. 모델의 확신 정도 보기
점수를 **확률(%)** 로 바꿔 봅시다.


In [ ]:
probabilities = torch.softmax(model(test_images[index].reshape(1, 64)), dim=1)
plt.bar(range(10), probabilities[0].detach())
plt.xticks(range(10))
plt.show()

## 다음 시간 예고
오늘 모델은 이미지를 **한 줄로 펼쳐서** 봤습니다. 그래서 숫자가 조금만 옆으로 밀려도 헷갈릴 수 있습니다.
5주차에는 이미지를 펼치지 않고 **작은 돋보기로 훑어보는** CNN을 배웁니다.
